# Which CT-to-3-channel strategy (and which backbone) best recognises abnormality patterns?

**Question.** DINOv2-S / DINOv3-S are standard 3-channel vision backbones; a chest CT slice is single-channel HU, so a strategy is needed to fill those 3 channels. DALE-CT-0, by contrast, is a CT-specific backbone that takes a single HU-windowed channel natively. This notebook compares **9 methods** purely on how well their frozen features support recognising the 18 CT-RATE abnormality labels - no domain-gap analysis and no compute-cost comparison here (see `CT_Domain_Gap_Analysis.ipynb` for those).

| # | Method | Channels | Backbone |
|---|---|---|---|
| 1-4 | `same` / `windows` / `adjacent` / `window_edge` | 3 (see below) | DINOv2-S |
| 5-8 | `same` / `windows` / `adjacent` / `window_edge` | 3 (see below) | DINOv3-S |
| 9 | `dale` (native) | 1 (DALE-CT-0's own HU window) | DALE-CT-0 (ViT-L, self-supervised on CT-RATE) |

3-channel strategies (all 4 use the exact same sampled slice indices per scan, so they are paired):
* `same` - one lung-windowed slice, replicated x3
* `windows` - the same slice through 3 clinical HU windows (lung / mediastinal / bone)
* `adjacent` - 3 consecutive slices (i-1, i, i+1), each lung-windowed
* `window_edge` - lung window, mediastinal window, Sobel edge-magnitude of the lung window

All 9 methods sample slices with the **same** slab logic (`CCE_SLAB_MM` mm per slice, default 12mm), so every method is compared on the same anatomical positions.

**Evaluation**: ~`N_CTRATE` CT-RATE validation scans (stratified by vendor x kernel, same selection code as `CT_Domain_Gap_Analysis.ipynb`) with their official 18-label annotations. A cross-validated logistic-regression probe and a k-NN probe (both frozen-feature, standard self-supervised-evaluation protocol) give macro AUROC per method; the **same** `GroupKFold` splits (by patient) are reused for every method and every label, so the comparison is paired, and a Wilcoxon/bootstrap/FDR pipeline tells you which differences are real.

### Setup
* Hugging Face token (secret `HF_TOKEN` on Colab, or `HF_TOKEN` env var) with access to the gated `ibrahimhamamci/CT-RATE` dataset. DINOv3 weights are also gated (accept the licence on its model page first). DINOv2 and DALE-CT-0 weights are public.
* GPU runtime recommended. Everything is cached under `channel_encoding_outputs/cache`, so re-runs are quick.
* Set `DRY_RUN_RANDOM_WEIGHTS = True` (or env `CCE_DRY_RUN=1`) to test the plumbing with random weights (results meaningless). **Tip**: for a quick local test, also set `CCE_N_CTRATE` to a small number - `huggingface_hub` can silently reuse a cached login token even with `HF_TOKEN` unset, so "no token" is not a reliable way to skip CT-RATE downloads.

**Licences:** CT-RATE CC-BY-NC-SA-4.0 (research use only). DINOv2 Apache-2.0. DALE-CT weights CC-BY-NC-SA-4.0. DINOv3 is released under Meta's own licence - read it before using results outside research.

## 1. Configuration

In [1]:
import os, sys, re, gc, json, time, shutil, warnings, datetime as dt
from pathlib import Path
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
def _env(name, default=None, cast=str):
    v = os.environ.get(name)
    return default if v in (None, "") else cast(v)

# ------------------------------ USER SETTINGS ------------------------------
N_CTRATE = _env("CCE_N_CTRATE", 200, int)                # CT-RATE validation scans (unique patients), stratified by vendor x kernel
N_CTRATE_PAIRS = 0                                        # sibling reconstructions are not used in this notebook
CTRATE_SEED = 0
CTRATE_MAX_VOXELS = 512 * 512 * 700                       # RAM guard (12 GB Colab)
DEVICE = "auto"                                           # "auto" -> cuda if available

DINO_MODELS = {                                           # timm checkpoint ids (confirmed to exist in timm 1.0.30)
    "dino2": _env("CCE_DINOV2_MODEL", "vit_small_patch14_dinov2.lvd142m"),    # public, Apache-2.0
    "dino3": _env("CCE_DINOV3_MODEL", "vit_small_patch16_dinov3.lvd1689m"),   # gated (Meta's DINOv3 licence) - accept it on the HF model page, then set HF_TOKEN
}
STRATEGIES = ("same", "windows", "adjacent", "window_edge")
DALE_MODEL = _env("CCE_DALE_MODEL", "Kentucky-Open-Science/DALE-CT-0")        # 1-channel, self-supervised on CT-RATE; public

CCE_SLAB_MM = _env("CCE_SLAB_MM", 12.0, float)            # one slice per N mm slab - SAME for every method (DINO strategies and DALE), so slice positions are identical
CCE_MAX_SLICES = _env("CCE_MAX_SLICES", None, int)        # cap sampled slices per scan (quick tests)
CCE_BATCH_SLICES = _env("CCE_BATCH_SLICES", 16, int)      # slices per backbone forward-pass chunk

PROBE_FOLDS = 5                                           # group k-fold splits (by patient), shared by every method and every label
MIN_POS = 5                                               # a label needs >= this many positives (and negatives) to be probed at all
N_BOOT = 2000

DELETE_RAW_AFTER_EMBED = True
DRY_RUN_RANDOM_WEIGHTS = _env("CCE_DRY_RUN", "0") == "1"
HF_TOKEN = os.environ.get("HF_TOKEN")
# ---------------------------------------------------------------------------

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = "/content/drive/MyDrive/NHRD - Local Sample"
    WORK_DIR = "/content/channel_encoding_work"
    if not HF_TOKEN:
        try:
            from google.colab import userdata
            HF_TOKEN = userdata.get("HF_TOKEN")
        except Exception:
            pass
else:
    BASE_DIR = _env("CCE_BASE_DIR", os.path.expanduser("~/NHRD - Local Sample"))
    WORK_DIR = _env("CCE_WORK_DIR", os.path.expanduser("~/channel_encoding_work"))

OUTPUT_DIR = Path(_env("CCE_OUTPUT_DIR", f"{BASE_DIR}/channel_encoding_outputs")) / ("dryrun" if DRY_RUN_RANDOM_WEIGHTS else "")
CACHE_DIR = OUTPUT_DIR / "cache"
RAW_DIR = Path(WORK_DIR) / "raw"
WEIGHTS_DIR = Path(WORK_DIR) / "weights"
for d in (OUTPUT_DIR / "figures", CACHE_DIR, RAW_DIR, WEIGHTS_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("Colab:", IN_COLAB, "| dry run:", DRY_RUN_RANDOM_WEIGHTS, "| N_CTRATE:", N_CTRATE)
for k in ("BASE_DIR", "OUTPUT_DIR", "WORK_DIR"):
    print(f"{k:11s}", globals()[k])

Mounted at /content/drive
Colab: True | dry run: False | N_CTRATE: 200
BASE_DIR    /content/drive/MyDrive/NHRD - Local Sample
OUTPUT_DIR  /content/drive/MyDrive/NHRD - Local Sample/channel_encoding_outputs
WORK_DIR    /content/channel_encoding_work


## 2. Environment
No CT-CLIP/CTViT repo and no DICOM reading needed here - this notebook only reads CT-RATE NIfTI volumes (already in the training arrangement) and runs `timm` backbones (DINOv2-S, DINOv3-S) plus DALE-CT-0.

In [2]:
def pip_install(*pkgs):
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

try:
    import torch
except ImportError:
    pip_install("torch"); import torch
pip_install("nibabel", "huggingface_hub", "timm", "safetensors", "scikit-learn", "scipy", "requests", "pandas", "matplotlib")

import numpy as np, pandas as pd, requests
import torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
warnings.filterwarnings("ignore")

DEV = ("cuda" if torch.cuda.is_available() else "cpu") if DEVICE == "auto" else DEVICE
print("torch", torch.__version__, "| device:", DEV, "| GPU:", torch.cuda.get_device_name(0) if DEV == "cuda" else "-", "| threads:", torch.get_num_threads())
if HF_TOKEN:
    try:
        from huggingface_hub import login
        login(token=HF_TOKEN, add_to_git_credential=False)
    except Exception as e:
        print("huggingface_hub login failed (gated DINOv3/CT-RATE downloads may fail):", repr(e)[:200])

torch 2.11.0+cu128 | device: cuda | GPU: Tesla T4 | threads: 1


## 3. Reading CT-RATE volumes
CT-RATE NIfTI arrays are already stored in the arrangement CT-CLIP/DALE-CT-0 were trained on, so no DICOM/orientation handling is needed here (unlike the domain-gap notebook, which also ingests raw local DICOMs). **Reused verbatim from `CT_Domain_Gap_Analysis.ipynb`**: `_crop_pad_axis`, `preprocess_volume`, `TARGET_SPACING`/`TARGET_SHAPE`.

In [3]:
TARGET_SPACING = (1.5, 0.75, 0.75)      # (z, height, width) mm
TARGET_SHAPE = (240, 480, 480)          # voxels

def _crop_pad_axis(t, dim, target):
    n = t.shape[dim]
    start = max((n - target) // 2, 0)
    t = t.narrow(dim, start, min(start + target, n) - start)
    before = (target - t.shape[dim]) // 2
    return t, (before, target - t.shape[dim] - before)

def preprocess_volume(vol_B, spacing_B):
    """Official CT-CLIP preprocessing of an already CT-RATE-arranged HU volume -> float32 (1, 240, 480, 480), air/padding = -1."""
    v = torch.from_numpy(np.clip(vol_B, -1000, 1000))[None, None]
    new_shape = [int(n * cur / tgt) for n, cur, tgt in zip(v.shape[2:], spacing_B, TARGET_SPACING)]
    v = F.interpolate(v, size=new_shape, mode="trilinear", align_corners=False)[0, 0] / 1000.0
    pads = []
    for dim, tgt in enumerate(TARGET_SHAPE):
        v, p = _crop_pad_axis(v, dim, tgt)
        pads.append(p)
    v = F.pad(v, (*pads[2], *pads[1], *pads[0]), value=-1.0)
    assert tuple(v.shape) == TARGET_SHAPE, v.shape
    return v.unsqueeze(0).contiguous()

## 4. CT-RATE scan selection
**Reused verbatim**: `select_ctrate` (stratified vendor x kernel round-robin), `hf_fetch`/`ctrate_rel_path`/`download_ctrate`, `vendor_group`/`kernel_family`, `read_ctrate_volume`. Only the `ctrate` cohort is used in this notebook (no local/LIDC scans - they played no role once domain-gap analysis was taken out of scope).

In [4]:
CT_REPO = "ibrahimhamamci/CT-RATE"
CTRATE_DIR = Path(WORK_DIR) / "ctrate_meta"
CTRATE_DIR.mkdir(parents=True, exist_ok=True)
_num = lambda s: (re.findall(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", str(s)) or ["nan"])[0]

def hf_fetch(rel, local_dir):
    dst = Path(local_dir) / rel
    if dst.exists() and dst.stat().st_size > 0:
        return dst
    from huggingface_hub import hf_hub_download
    return Path(hf_hub_download(repo_id=CT_REPO, repo_type="dataset", filename=rel, token=HF_TOKEN or None, local_dir=str(local_dir)))

def ctrate_rel_path(name):
    m = re.match(r"valid_(\d+)_([a-z]+)_(\d+)\.nii\.gz$", name)
    return f"dataset/valid_fixed/valid_{m[1]}/valid_{m[1]}_{m[2]}/{name}"

def vendor_group(s):
    s = str(s).lower()
    return "Philips" if "philips" in s else "Siemens" if "siemens" in s else "PNMS" if "pnms" in s else "Toshiba" if ("toshiba" in s or "canon" in s) else "GE" if "ge " in s or s.startswith("ge") else str(s)

def kernel_family(k):
    k = str(k).strip("[]' ").split("'")[0].split(",")[0].strip()
    return k or "?"

def select_ctrate(meta, labels_idx, nochest, n_total, n_pairs, seed, max_voxels):
    """Stratified (vendor x kernel) round-robin selection of unique patients. Returns (chosen_names, sibling_map, meta_df)."""
    M = meta.copy()
    M["xy"] = M.XYSpacing.map(lambda s: float(_num(s))); M["z"] = pd.to_numeric(M.ZSpacing, errors="coerce")
    M["voxels"] = M.Rows * M.Columns * M.NumberofSlices; M["zext"] = M.NumberofSlices * M.z
    M["patient"] = M.index.str.extract(r"valid_(\d+)_", expand=False); M["acq"] = M.index.str.extract(r"valid_(\d+_[a-z]+)_", expand=False)
    M["vendor"] = M.Manufacturer.map(vendor_group); M["kfam"] = M.ConvolutionKernel.map(kernel_family)
    ok = M[~M.index.str.replace(".nii.gz", "", regex=False).isin(nochest) & M.z.notna() & M.xy.notna() & (M.voxels <= max_voxels) & (M.zext >= 150) & M.index.isin(labels_idx)]
    rng = np.random.default_rng(seed)
    pool = ok.sample(frac=1.0, random_state=seed).sort_values("voxels", kind="stable").groupby("patient").head(1)
    strata = {k: list(g.index) for k, g in pool.groupby(["vendor", "kfam"])}
    for k in strata:
        rng.shuffle(strata[k])
    order = sorted(strata, key=lambda k: -len(strata[k]))
    chosen = []
    while len(chosen) < n_total and any(strata.values()):
        for k in order:
            if strata[k] and len(chosen) < n_total:
                chosen.append(strata[k].pop())
    sib = {}
    if n_pairs:
        cand = [n for n in chosen if len(ok[(ok.acq == M.loc[n, "acq"]) & (ok.index != n)]) > 0]
        rng.shuffle(cand)
        for n in cand[:n_pairs]:
            others = ok[(ok.acq == M.loc[n, "acq"]) & (ok.index != n)]
            sib[n] = others.sort_values("voxels").index[0]
    return sorted(chosen), sib, M

PATHOLOGIES = ['Medical material', 'Arterial wall calcification', 'Cardiomegaly', 'Pericardial effusion', 'Coronary artery wall calcification', 'Hiatal hernia', 'Lymphadenopathy', 'Emphysema',
               'Atelectasis', 'Lung nodule', 'Lung opacity', 'Pulmonary fibrotic sequela', 'Pleural effusion', 'Mosaic attenuation pattern', 'Peribronchial thickening', 'Consolidation',
               'Bronchiectasis', 'Interlobular septal thickening']

ITEMS = []
CT_META = ct_labels = None
try:
    ct_meta = pd.read_csv(hf_fetch("dataset/metadata/validation_metadata.csv", CTRATE_DIR)).set_index("VolumeName")
    ct_labels = pd.read_csv(hf_fetch("dataset/multi_abnormality_labels/valid_predicted_labels.csv", CTRATE_DIR)).set_index("VolumeName")
    ct_nochest = set(re.findall(r"(valid_\d+_[a-z]+_\d+)\.nii\.gz", hf_fetch("dataset/metadata/no_chest_valid.txt", CTRATE_DIR).read_text()))
    prim, sibs, CT_META = select_ctrate(ct_meta, set(ct_labels.index), ct_nochest, N_CTRATE, N_CTRATE_PAIRS, CTRATE_SEED, CTRATE_MAX_VOXELS)
    for n in prim:
        pid = re.match(r"valid_(\d+)_", n)[1]
        ITEMS.append(dict(cohort="ctrate", id=n.replace(".nii.gz", ""), name=n, group=f"ctrate_{pid}", primary=True))
    sel = CT_META.loc[prim]
    print(f"CT-RATE: {len(prim)} unique patients selected")
    display(pd.crosstab(sel.vendor, sel.kfam))
except Exception as e:
    print("CT-RATE could not be set up:", repr(e)[:300], "\n-> check HF_TOKEN and that you accepted the dataset terms. This notebook cannot run without it.")

print("total volumes to process:", len(ITEMS))

validation_metadata.csv:   0%|          | 0.00/948k [00:00<?, ?B/s]

valid_predicted_labels.csv:   0%|          | 0.00/174k [00:00<?, ?B/s]

no_chest_valid.txt:   0%|          | 0.00/1.83k [00:00<?, ?B/s]

CT-RATE: 200 unique patients selected


kfam,A,B,Bl56f,Bl57d,Br36d,Br40f,Br44f,Br60f,EA,L,SA,SB
vendor,,,,,,,,,,,,
PNMS,0,0,0,0,0,0,0,0,23,0,9,1
Philips,23,24,0,0,0,0,0,0,0,1,0,0
Siemens,0,0,23,23,23,23,4,23,0,0,0,0


total volumes to process: 200


In [5]:
def read_ctrate_volume(name, raw_dir):
    """CT-RATE NIfTI -> (vol (k, i, j) = (Z, LEFT axis, POSTERIOR axis) HU float32, spacing, meta)."""
    import nibabel as nib
    nii = nib.load(str(Path(raw_dir) / ctrate_rel_path(name)))
    vol = nii.get_fdata().astype(np.float32)
    row = CT_META.loc[name]
    frac_air = float(((vol > -1050) & (vol < -950)).mean())
    if frac_air < 0.001:
        vol = vol * float(row.RescaleSlope) + float(row.RescaleIntercept)
    xy, z = float(_num(row.XYSpacing)), float(row.ZSpacing)
    meta = dict(vendor=vendor_group(row.Manufacturer), kernel=kernel_family(row.ConvolutionKernel), z_spacing=z, xy_spacing=xy)
    return vol.transpose(2, 0, 1), (z, xy, xy), meta

def download_ctrate(name, raw_dir):
    return hf_fetch(ctrate_rel_path(name), raw_dir)

## 5. Channel-construction strategies and the two extractors
All 9 methods start from the **same sampled slice indices** per scan (one slice per `CCE_SLAB_MM` mm) via `slab_slice_idx`, so the comparison is paired - differences come only from how each method represents those slices, not from which anatomy was sampled.

| Strategy | Channel 1 | Channel 2 | Channel 3 |
|---|---|---|---|
| `same` | lung window | lung window (copy) | lung window (copy) |
| `windows` | lung window | mediastinal window | bone window |
| `adjacent` | slice i-1 (lung window) | slice i (lung window) | slice i+1 (lung window) |
| `window_edge` | lung window | mediastinal window | Sobel edge-magnitude of the lung window |

Windows are standard clinical HU windows (center, width): lung (-600, 1500), mediastinal (40, 400), bone (400, 1800), each mapped to [0,1]. DALE-CT-0 instead uses its own 1-channel preprocessing exactly as published (HU clipped to [-997, 888], min-max then z-scored with its model card's mean/std) - it is not forced through the clinical windows above, since the point is to compare it as its own native method.

In [6]:
WINDOWS = {"lung": (-600.0, 1500.0), "mediastinal": (40.0, 400.0), "bone": (400.0, 1800.0)}
_SOBEL_X = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]])
_SOBEL_Y = _SOBEL_X.t()

def window_hu(hu, center, width):
    lo = center - width / 2.0
    return ((hu - lo) / width).clamp(0.0, 1.0)

def sobel_edge_map(img):
    """img: (N,1,H,W) in [0,1] -> edge magnitude in [0,1], normalised per-slice by its own 99th percentile."""
    gx = F.conv2d(F.pad(img, (1, 1, 1, 1), mode="replicate"), _SOBEL_X.view(1, 1, 3, 3))
    gy = F.conv2d(F.pad(img, (1, 1, 1, 1), mode="replicate"), _SOBEL_Y.view(1, 1, 3, 3))
    mag = torch.sqrt(gx * gx + gy * gy + 1e-12)
    q = torch.quantile(mag.flatten(1), 0.99, dim=1).clamp_min(1e-6).view(-1, 1, 1, 1)
    return (mag / q).clamp(0.0, 1.0)

def slab_slice_idx(x, slab_mm=CCE_SLAB_MM, max_slices=CCE_MAX_SLICES, spacing_mm=TARGET_SPACING[0]):
    """x: (Z,H,W) model-input tensor, HU/1000 in [-1,1], padding=-1. -> (idx, valid): idx = one slice per slab_mm, centred; valid = all non-padding slice indices.
    Shared by every method in this notebook (DINO strategies and DALE), so all 9 feature spaces are compared on identical anatomical positions."""
    valid = torch.nonzero(x.amax(dim=(1, 2)) > -0.99).flatten()
    step = max(1, int(round(slab_mm / spacing_mm)))
    idx = valid[step // 2::step]
    if max_slices and len(idx) > max_slices:
        idx = idx[np.linspace(0, len(idx) - 1, max_slices).astype(int)]
    return idx, valid

def build_channels(x, idx, valid, strategy):
    """x: (Z,H,W) tensor in [-1,1] (HU/1000). -> (S,3,H,W) in [0,1], one of the 4 DINO-side strategies."""
    hu = x[idx] * 1000.0
    lung = window_hu(hu, *WINDOWS["lung"])
    if strategy == "same":
        return torch.stack([lung, lung, lung], dim=1)
    if strategy == "windows":
        med = window_hu(hu, *WINDOWS["mediastinal"]); bone = window_hu(hu, *WINDOWS["bone"])
        return torch.stack([lung, med, bone], dim=1)
    if strategy == "adjacent":
        lo, hi = int(valid[0]), int(valid[-1])
        idx_m1 = (idx - 1).clamp(lo, hi); idx_p1 = (idx + 1).clamp(lo, hi)
        lung_m1 = window_hu(x[idx_m1] * 1000.0, *WINDOWS["lung"]); lung_p1 = window_hu(x[idx_p1] * 1000.0, *WINDOWS["lung"])
        return torch.stack([lung_m1, lung, lung_p1], dim=1)
    if strategy == "window_edge":
        med = window_hu(hu, *WINDOWS["mediastinal"]); edge = sobel_edge_map(lung[:, None])[:, 0]
        return torch.stack([lung, med, edge], dim=1)
    raise ValueError(f"unknown strategy: {strategy}")

class DinoChannelExtractor:
    """Both DINO backbones x all 4 channel strategies. .embed(t, want) returns only the requested '{backbone}_{strategy}' keys."""
    name = "dino_channel"
    keys = tuple(f"{bk}_{st}" for bk in DINO_MODELS for st in STRATEGIES)

    def __init__(self, dev):
        import timm
        self.dev = dev
        self.models, self.norm, self.tgt_size = {}, {}, {}
        for bk, model_name in DINO_MODELS.items():
            try:
                if DRY_RUN_RANDOM_WEIGHTS:
                    m = timm.create_model(model_name, pretrained=False, num_classes=0, dynamic_img_size=True).eval()
                    torch.manual_seed(0); print(f"  {bk}: RANDOM weights (dry run) [{model_name}]")
                else:
                    m = timm.create_model(model_name, pretrained=True, num_classes=0, dynamic_img_size=True).eval()
                    print(f"  {bk}: pretrained weights loaded [{model_name}]")
                cfg = getattr(m, "pretrained_cfg", {}) or {}
                mean = torch.tensor(cfg.get("mean", (0.485, 0.456, 0.406))).view(3, 1, 1)
                std = torch.tensor(cfg.get("std", (0.229, 0.224, 0.225))).view(3, 1, 1)
                patch = m.patch_embed.patch_size[0]                          # dynamic_img_size still requires H/W divisible by the patch size
                self.tgt_size[bk] = (TARGET_SHAPE[1] // patch) * patch       # nearest multiple of patch size <= 480 (476 for patch14, 480 for patch16)
                m.to(dev)
                self.models[bk] = m; self.norm[bk] = (mean, std)
            except Exception as e:
                print(f"  {bk} FAILED to load and is dropped: {repr(e)[:300]}")
        assert self.models, "no DINO backbone could be loaded"

    @torch.no_grad()
    def embed(self, t, want=None):
        want = set(want) if want is not None else set(self.keys)
        x = t[0]
        idx, valid = slab_slice_idx(x)
        out = {}
        for strategy in STRATEGIES:
            need_bks = [bk for bk in self.models if f"{bk}_{strategy}" in want]
            if not need_bks:
                continue
            ch = build_channels(x, idx, valid, strategy)                        # (S,3,H,W) in [0,1], CPU
            for bk in need_bks:
                size = self.tgt_size[bk]
                ch_bk = F.interpolate(ch, size=(size, size), mode="bilinear", align_corners=False) if size != ch.shape[-1] else ch
                mean, std = self.norm[bk]
                z = (ch_bk - mean) / std
                outs = []
                for i in range(0, len(z), CCE_BATCH_SLICES):
                    b = z[i:i + CCE_BATCH_SLICES].to(self.dev)
                    with torch.autocast("cuda", dtype=torch.float16, enabled=(self.dev == "cuda")):
                        outs.append(self.models[bk](b).float().cpu())
                out[f"{bk}_{strategy}"] = torch.cat(outs).mean(0).numpy()
        return out

def load_safetensors_strict(model, path, name):
    from safetensors.torch import load_file
    res = model.load_state_dict(load_file(str(path)), strict=True)
    print(f"  {name}: weights loaded strictly ({len(model.state_dict())} tensors)")

def hub_file(repo, filename, repo_type="model", token=None):
    from huggingface_hub import hf_hub_download
    return Path(hf_hub_download(repo_id=repo, filename=filename, repo_type=repo_type, token=token, local_dir=str(WEIGHTS_DIR / repo.replace("/", "__"))))

class DaleExtractor:
    """DALE-CT-0's own 1-channel method: same sampled slice positions (slab_slice_idx), its own HU window/normalisation, no clinical-window substitution."""
    name, keys = "dale", ("dale",)
    CLIP_MIN, CLIP_MAX, MEAN_HU, STD_HU = -997.0, 888.0, -142.39, 360.97

    def __init__(self, dev, repo=None):
        import timm
        repo = repo or DALE_MODEL
        self.dev = dev
        self.m = timm.create_model("vit_large_patch14_dinov2", pretrained=False, num_classes=0, in_chans=1, patch_size=16, img_size=512, dynamic_img_size=True).eval()
        if DRY_RUN_RANDOM_WEIGHTS:
            torch.manual_seed(0); print("  dale: RANDOM weights (dry run)")
        else:
            load_safetensors_strict(self.m, hub_file(repo, "model.safetensors"), "dale")
        self.m.to(dev)
        rng = self.CLIP_MAX - self.CLIP_MIN
        self.nm, self.ns = (self.MEAN_HU - self.CLIP_MIN) / rng, self.STD_HU / rng

    @torch.no_grad()
    def embed(self, t, want=None):
        if want is not None and "dale" not in want:
            return {}
        x = t[0]
        idx, valid = slab_slice_idx(x)                                        # same shared slab logic as the DINO strategies
        hu = (x[idx] * 1000.0).clamp(self.CLIP_MIN, self.CLIP_MAX)
        z = (((hu - self.CLIP_MIN) / (self.CLIP_MAX - self.CLIP_MIN)) - self.nm) / self.ns
        outs = []
        for i in range(0, len(z), CCE_BATCH_SLICES):
            b = z[i:i + CCE_BATCH_SLICES, None].to(self.dev)
            with torch.autocast("cuda", dtype=torch.float16, enabled=(self.dev == "cuda")):
                outs.append(self.m(b).float().cpu())
        return {"dale": torch.cat(outs).mean(0).numpy()}

EXT = {}
for name, cls in (("dino", DinoChannelExtractor), ("dale", DaleExtractor)):
    try:
        EXT[name] = cls(DEV)
    except Exception as e:
        print(f"extractor {name} FAILED to load and is dropped: {repr(e)[:300]}")
assert EXT, "no extractor could be loaded"
KEYS_NEEDED = [k for e in EXT.values() for k in e.keys]
SPACE_OF = {k: (f"{'DINOv2-S' if k.startswith('dino2') else 'DINOv3-S'} / {k.split('_', 1)[1]}" if k != "dale" else "DALE-CT-0 (1-channel, native)") for k in KEYS_NEEDED}
print("methods under test:", KEYS_NEEDED)

model.safetensors: reconstructing file:   0%|          |  0.00B / 88.2MB            

model.safetensors: downloading bytes:           |  0.00B            

  dino2: pretrained weights loaded [vit_small_patch14_dinov2.lvd142m]


model.safetensors: reconstructing file:   0%|          |  0.00B / 86.4MB            

model.safetensors: downloading bytes:           |  0.00B            

  dino3: pretrained weights loaded [vit_small_patch16_dinov3.lvd1689m]


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.21GB            

model.safetensors: downloading bytes:           |  0.00B            

  dale: weights loaded strictly (342 tensors)
methods under test: ['dino2_same', 'dino2_windows', 'dino2_adjacent', 'dino2_window_edge', 'dino3_same', 'dino3_windows', 'dino3_adjacent', 'dino3_window_edge', 'dale']


### Unit check: the strategies actually differ
Regression guard, run before any real extraction: `same`'s 3 channels must be identical by construction, and the other 3 DINO-side strategies must NOT be (otherwise a bug silently built 3 identical channels).

In [7]:
def _unit_check_channels():
    rng = torch.Generator().manual_seed(0)
    x = torch.rand((40, 32, 32), generator=rng) * 1.6 - 1.0           # fake volume in roughly [-1, 0.6]
    x[:6] = -1.0; x[-6:] = -1.0                                        # padding at both ends
    idx, valid = slab_slice_idx(x, slab_mm=12.0, max_slices=None)
    assert len(idx) >= 1, "no slices sampled in the unit check"
    for strat in STRATEGIES:
        ch = build_channels(x, idx, valid, strat)
        d01 = (ch[:, 0] - ch[:, 1]).abs().max().item()
        d02 = (ch[:, 0] - ch[:, 2]).abs().max().item()
        if strat == "same":
            assert d01 == 0.0 and d02 == 0.0, f"'same' strategy channels are not identical ({d01}, {d02})"
        else:
            assert d01 > 0.0 or d02 > 0.0, f"'{strat}' strategy produced 3 identical channels (likely a bug)"
    print("unit check OK: 'same' channels are identical; 'windows'/'adjacent'/'window_edge' channels differ")

_unit_check_channels()

unit check OK: 'same' channels are identical; 'windows'/'adjacent'/'window_edge' channels differ


## 6. Extraction loop (cached)
Read a CT-RATE volume (downloading on demand) -> preprocess -> every method's embedding, cached to `channel_encoding_outputs/cache/<id>.npz`. Re-running skips finished scans; adding a method later only computes the new keys.

In [8]:
PIPE_VERSION = "v1"
def _pack(**kw):
    return {k: (np.array(json.dumps(v)) if isinstance(v, (dict, list)) else np.asarray(v)) for k, v in kw.items()}

def load_cache(it):
    p = CACHE_DIR / f"{it['id']}.npz"
    if not p.exists():
        return {}
    d = dict(np.load(p, allow_pickle=False))
    return d if str(d.get("pipe", "")) == PIPE_VERSION else {}

def obtain_volume(it):
    p = download_ctrate(it["name"], RAW_DIR)
    vB, spB, meta = read_ctrate_volume(it["name"], RAW_DIR)
    return vB, spB, meta, p

def process_item(it):
    have = load_cache(it)
    need = [k for k in KEYS_NEEDED if k not in have]
    if not need and "meta" in have:
        return have
    vB, spB, meta, cleanup = obtain_volume(it)
    t = preprocess_volume(vB, spB); del vB
    out = dict(have)
    for e in EXT.values():
        if any(k in need for k in e.keys):
            for k, v in e.embed(t, want=need).items():
                out[k] = np.asarray(v, dtype=np.float32)
    out.update(_pack(meta={**meta, "cohort": it["cohort"]}, pipe=PIPE_VERSION))
    np.savez_compressed(CACHE_DIR / f"{it['id']}.npz", **out)
    if cleanup is not None and DELETE_RAW_AFTER_EMBED:
        try:
            shutil.rmtree(cleanup) if Path(cleanup).is_dir() else Path(cleanup).unlink()
        except Exception:
            pass
    return out

FAILED = {}
t_all = time.time()
for n, it in enumerate(ITEMS, 1):
    try:
        t0 = time.time(); process_item(it)
        print(f"[{n}/{len(ITEMS)}] {it['id']:<34s} {time.time() - t0:5.1f}s")
    except Exception as e:
        FAILED[it["id"]] = repr(e)[:200]; print(f"[{n}/{len(ITEMS)}] {it['id']:<34s} FAILED: {FAILED[it['id']]}")
    gc.collect()
    if DEV == "cuda":
        torch.cuda.empty_cache()
print(f"done in {(time.time() - t_all) / 60:.1f} min | failed: {len(FAILED)}")

dataset/valid_fixed/valid_1009/valid_100(…): reconstructing file:   0%|          |  0.00B /  164MB            

dataset/valid_fixed/valid_1009/valid_100(…): downloading bytes:           |  0.00B            

[1/200] valid_1009_a_1                      16.7s


dataset/valid_fixed/valid_1016/valid_101(…): reconstructing file:   0%|          |  0.00B / 73.1MB            

dataset/valid_fixed/valid_1016/valid_101(…): downloading bytes:           |  0.00B            

...

dataset/valid_fixed/valid_977/valid_977_(…): downloading bytes:           |  0.00B            

[199/200] valid_977_a_2                        9.4s


dataset/valid_fixed/valid_985/valid_985_(…): reconstructing file:   0%|          |  0.00B / 63.7MB            

dataset/valid_fixed/valid_985/valid_985_(…): downloading bytes:           |  0.00B            

[200/200] valid_985_a_1                        6.5s
done in 29.7 min | failed: 0


In [9]:
# ---- assemble tables from the cache
rows, EMB = [], {k: {} for k in KEYS_NEEDED}
for it in ITEMS:
    c = load_cache(it)
    if not c or "meta" not in c or any(k not in c for k in KEYS_NEEDED):
        continue
    meta = json.loads(str(c["meta"]))
    rows.append(dict(id=it["id"], cohort=it["cohort"], group=it["group"], name=it.get("name"), vendor=vendor_group(meta.get("vendor")), kernel=kernel_family(meta.get("kernel"))))
    for k in EMB:
        EMB[k][it["id"]] = c[k]
META_DF = pd.DataFrame(rows).set_index("id") if rows else pd.DataFrame(columns=["cohort", "group", "name", "vendor", "kernel"], index=pd.Index([], name="id"))
POS = {i: n for n, i in enumerate(META_DF.index)}
EMB = {k: np.vstack([v[i] for i in META_DF.index]) for k, v in EMB.items() if v}
print("scans with complete features:", len(META_DF))
if len(META_DF):
    display(pd.crosstab(META_DF.vendor, META_DF.kernel, margins=True))

scans with complete features: 200


kernel,A,B,Bl56f,Bl57d,Br36d,Br40f,Br44f,Br60f,EA,L,SA,SB,All
vendor,,,,,,,,,,,,,
PNMS,0,0,0,0,0,0,0,0,23,0,9,1,33
Philips,23,24,0,0,0,0,0,0,0,1,0,0,48
Siemens,0,0,23,23,23,23,4,23,0,0,0,0,119
All,23,24,23,23,23,23,4,23,23,1,9,1,200


## 7. Statistics helpers
Only what the paired comparison in sections 8-9 needs (no domain-gap / novelty toolbox here). `bh_fdr` is reused verbatim from `CT_Domain_Gap_Analysis.ipynb`; `paired_bootstrap_ci` is a small addition in the same style, for a paired (not independent-sample) bootstrap.

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from scipy.stats import wilcoxon
from itertools import combinations

RNG = np.random.default_rng(12345)

def bh_fdr(p):
    p = np.asarray(p, float); n = len(p); o = np.argsort(p); q = np.empty(n); prev = 1.0
    for rank, i in zip(range(n, 0, -1), o[::-1]):
        prev = min(prev, p[i] * n / rank); q[i] = prev
    return q

def paired_bootstrap_ci(diffs, n_boot, rng):
    """Bootstrap CI on the MEAN of an already-paired difference vector (e.g. per-label AUROC_a - AUROC_b)."""
    diffs = diffs[~np.isnan(diffs)]
    if len(diffs) == 0:
        return (np.nan, np.nan)
    vals = [np.mean(rng.choice(diffs, len(diffs), replace=True)) for _ in range(n_boot)]
    return tuple(np.nanpercentile(vals, [2.5, 97.5]))

## 8. Linear-probe / kNN-probe evaluation on CT-RATE labels (primary, and only, evidence)
For every one of the 9 methods: a cross-validated logistic-regression probe (macro AUROC across the 18 labels) and a k-NN probe (training-free complement, the standard self-supervised-evaluation protocol). The **same** `GroupKFold` splits (grouped by patient) are reused for every method and every label, so the comparison across methods is paired. A label is only probed if it has at least `MIN_POS` positives and `MIN_POS` negatives in the whole labelled set.

In [11]:
def build_label_matrix(meta_df, ct_labels, items):
    names = {it["id"]: it.get("name") for it in items}
    ids = [i for i in meta_df.index if names.get(i) in ct_labels.index]
    Y = ct_labels.loc[[names[i] for i in ids], PATHOLOGIES].to_numpy(float)
    groups = meta_df.loc[ids, "group"].to_numpy()
    return ids, Y, groups

def linear_probe_auroc(X, Y, folds, min_pos=MIN_POS):
    aurocs = np.full(Y.shape[1], np.nan)
    for j in range(Y.shape[1]):
        y = Y[:, j]
        if y.sum() < min_pos or (len(y) - y.sum()) < min_pos:
            continue
        preds = np.full(len(y), np.nan); ok = True
        for tr, te in folds:
            if len(np.unique(y[tr])) < 2:
                ok = False; break
            sc = StandardScaler().fit(X[tr])
            clf = LogisticRegression(C=1.0, class_weight="balanced", max_iter=2000).fit(sc.transform(X[tr]), y[tr])
            preds[te] = clf.decision_function(sc.transform(X[te]))
        if ok:
            try:
                aurocs[j] = roc_auc_score(y, preds)
            except Exception:
                pass
    return aurocs

def knn_probe_auroc(X, Y, folds, k=15, min_pos=MIN_POS):
    aurocs = np.full(Y.shape[1], np.nan)
    for j in range(Y.shape[1]):
        y = Y[:, j]
        if y.sum() < min_pos or (len(y) - y.sum()) < min_pos:
            continue
        preds = np.full(len(y), np.nan)
        for tr, te in folds:
            sc = StandardScaler().fit(X[tr])
            clf = KNeighborsClassifier(n_neighbors=min(k, len(tr) - 1), weights="distance").fit(sc.transform(X[tr]), y[tr])
            p = clf.predict_proba(sc.transform(X[te]))
            preds[te] = p[:, list(clf.classes_).index(1)] if 1 in clf.classes_ else 0.0
        try:
            aurocs[j] = roc_auc_score(y, preds)
        except Exception:
            pass
    return aurocs

RESULTS, PER_LABEL_AUC = [], {}
if CT_META is not None and len(EMB):
    PROBE_IDS, Y, GROUPS_P = build_label_matrix(META_DF, ct_labels, ITEMS)
    print(f"probe set: {len(PROBE_IDS)} CT-RATE scans with labels | {Y.shape[1]} candidate labels")
    if len(PROBE_IDS) >= 20:
        folds = list(GroupKFold(n_splits=PROBE_FOLDS).split(np.zeros(len(PROBE_IDS)), groups=GROUPS_P))
        rows_idx = [POS[i] for i in PROBE_IDS]
        for key, Xfull in EMB.items():
            X = Xfull[rows_idx]
            auc_lin = linear_probe_auroc(X, Y, folds)
            auc_knn = knn_probe_auroc(X, Y, folds)
            PER_LABEL_AUC[key] = auc_lin
            RESULTS.append(dict(key=key, space=SPACE_OF[key], macro_auroc_linear=float(np.nanmean(auc_lin)), macro_auroc_knn=float(np.nanmean(auc_knn)), n_labels_evaluated=int(np.sum(~np.isnan(auc_lin)))))
        RES_DF = pd.DataFrame(RESULTS).sort_values("macro_auroc_linear", ascending=False)
        display(Markdown("#### Macro AUROC across the 18 CT-RATE labels - how well does each method's frozen features recognise abnormality patterns?"))
        display(RES_DF.round(3))
        RES_DF.round(4).to_csv(OUTPUT_DIR / "probe_results.csv", index=False)
        LABEL_AUC_DF = pd.DataFrame(PER_LABEL_AUC, index=PATHOLOGIES)
        LABEL_AUC_DF.round(4).to_csv(OUTPUT_DIR / "per_label_auroc.csv")
        display(Markdown("#### Per-label AUROC (linear probe), methods as columns"))
        display(LABEL_AUC_DF.round(3))
    else:
        print("too few labelled CT-RATE scans for the probes (need >= 20) - increase N_CTRATE")
else:
    print("probe evaluation skipped (CT-RATE labels unavailable)")

probe set: 200 CT-RATE scans with labels | 18 candidate labels


#### Macro AUROC across the 18 CT-RATE labels - how well does each method's frozen features recognise abnormality patterns?

,key,space,macro_auroc_linear,macro_auroc_knn,n_labels_evaluated
2,dino2_adjacent,DINOv2-S / adjacent,0.675,0.625,18
1,dino2_windows,DINOv2-S / windows,0.674,0.616,18
0,dino2_same,DINOv2-S / same,0.673,0.618,18
6,dino3_adjacent,DINOv3-S / adjacent,0.665,0.602,18
7,dino3_window_edge,DINOv3-S / window_edge,0.663,0.600,18
8,dale,"DALE-CT-0 (1-channel, native)",0.663,0.611,18
5,dino3_windows,DINOv3-S / windows,0.648,0.607,18
4,dino3_same,DINOv3-S / same,0.647,0.604,18
3,dino2_window_edge,DINOv2-S / window_edge,0.639,0.596,18


#### Per-label AUROC (linear probe), methods as columns

,dino2_same,dino2_windows,dino2_adjacent,dino2_window_edge,dino3_same,dino3_windows,dino3_adjacent,dino3_window_edge,dale
Medical material,0.820,0.767,0.815,0.794,0.803,0.807,0.817,0.805,0.841
Arterial wall calcification,0.788,0.810,0.840,0.781,0.764,0.749,0.760,0.728,0.752
Cardiomegaly,0.785,0.774,0.802,0.716,0.763,0.774,0.754,0.798,0.815
Pericardial effusion,0.544,0.652,0.578,0.590,0.356,0.435,0.468,0.447,0.590
Coronary artery wall calcification,0.801,0.852,0.849,0.834,0.780,0.775,0.825,0.772,0.751
Hiatal hernia,0.575,0.631,0.555,0.560,0.507,0.475,0.528,0.621,0.649
Lymphadenopathy,0.463,0.501,0.537,0.493,0.565,0.559,0.568,0.625,0.586
Emphysema,0.688,0.562,0.690,0.607,0.617,0.691,0.638,0.609,0.614
Atelectasis,0.608,0.708,0.668,0.635,0.615,0.620,0.616,0.632,0.647
Lung nodule,0.503,0.531,0.538,0.428,0.552,0.535,0.515,0.562,0.556


## 9. Pairwise comparison across all 9 methods
For every pair of methods: the mean per-label AUROC difference with a paired bootstrap CI, a Wilcoxon signed-rank test across the 18 labels, and Benjamini-Hochberg correction across all pairwise comparisons together.

In [12]:
PAIR_ROWS = []
if len(PER_LABEL_AUC) >= 2:
    for a, b in combinations(PER_LABEL_AUC, 2):
        va, vb = PER_LABEL_AUC[a], PER_LABEL_AUC[b]
        mask = ~np.isnan(va) & ~np.isnan(vb)
        if mask.sum() < 5:
            continue
        diff = va[mask] - vb[mask]
        lo, hi = paired_bootstrap_ci(diff, N_BOOT, RNG)
        try:
            _, p = wilcoxon(va[mask], vb[mask])
        except Exception:
            p = np.nan
        PAIR_ROWS.append(dict(a=a, b=b, mean_auroc_diff=float(np.mean(diff)), ci_lo=lo, ci_hi=hi, n_labels=int(mask.sum()), wilcoxon_p=p))
    PAIR_DF = pd.DataFrame(PAIR_ROWS)
    if len(PAIR_DF):
        PAIR_DF["p_fdr"] = bh_fdr(PAIR_DF.wilcoxon_p.fillna(1.0).to_numpy())
        display(Markdown("#### Pairwise comparisons across all 9 methods (positive diff = `a` beats `b`; sorted by effect size)"))
        display(PAIR_DF.sort_values("mean_auroc_diff", key=lambda s: -s.abs()).round(4))
        PAIR_DF.round(4).to_csv(OUTPUT_DIR / "method_pairwise_stats.csv", index=False)
else:
    PAIR_DF = pd.DataFrame()
    print("pairwise comparison skipped (need probe results from section 8)")

#### Pairwise comparisons across all 9 methods (positive diff = `a` beats `b`; sorted by effect size)

,a,b,mean_auroc_diff,ci_lo,ci_hi,n_labels,wilcoxon_p,p_fdr
15,dino2_adjacent,dino2_window_edge,0.0365,0.0206,0.0529,18,0.0003,0.0118
9,dino2_windows,dino2_window_edge,0.0353,0.0187,0.0511,18,0.0010,0.0188
2,dino2_same,dino2_window_edge,0.0343,0.0083,0.0602,18,0.0342,0.3476
16,dino2_adjacent,dino3_same,0.0285,-0.0044,0.0628,18,0.1297,0.6329
10,dino2_windows,dino3_same,0.0273,-0.0151,0.0749,18,0.2645,0.6329
17,dino2_adjacent,dino3_windows,0.0272,-0.0024,0.0542,18,0.0432,0.3476
3,dino2_same,dino3_same,0.0263,-0.0048,0.0604,18,0.1674,0.6329
11,dino2_windows,dino3_windows,0.0260,-0.0146,0.0658,18,0.2462,0.6329
23,dino2_window_edge,dino3_adjacent,-0.0259,-0.0585,0.0062,18,0.1540,0.6329
4,dino2_same,dino3_windows,0.0249,-0.0044,0.0554,18,0.1540,0.6329


## 10. Summary

In [13]:
SUMMARY = {"n_ctrate": len(META_DF) if len(META_DF) else 0, "device": DEV, "dry_run": DRY_RUN_RANDOM_WEIGHTS, "created": dt.datetime.now().isoformat(timespec="seconds")}
lines = []
if RESULTS:
    best = pd.DataFrame(RESULTS).sort_values("macro_auroc_linear", ascending=False).reset_index(drop=True)
    top = best.iloc[0]
    lines.append(f"- **Best overall by macro linear-probe AUROC**: **{top.space}** ({top.macro_auroc_linear:.3f}, kNN {top.macro_auroc_knn:.3f}, {top.n_labels_evaluated} labels evaluated).")
    lines.append("- **Full ranking**: " + "; ".join(f"{r.space} {r.macro_auroc_linear:.3f}" for r in best.itertuples()))
    dale_row = best[best.key == "dale"]
    if len(dale_row):
        d = dale_row.iloc[0]
        rank = int(best.index[best.key == "dale"][0]) + 1
        lines.append(f"- **DALE-CT-0 (1-channel, native)** ranks {rank} of {len(best)} ({d.macro_auroc_linear:.3f}); compare this directly against the best 3-channel strategy above to judge whether the 3-channel construction helps at all over the native 1-channel method.")
    if len(PAIR_DF):
        sig = PAIR_DF[PAIR_DF.p_fdr < 0.05]
        lines.append(f"- {len(sig)} of {len(PAIR_DF)} pairwise comparisons are significant after FDR correction (see `method_pairwise_stats.csv` for effect sizes and CIs).")
    lines.append("- Caution: labels are CT-RATE's own (automated) annotations, not independent ground truth. This notebook measures frozen-feature probe performance only - it says nothing about compute cost or how these methods generalise to a different scanner (see `CT_Domain_Gap_Analysis.ipynb` for that).")
else:
    lines.append("- No probe results (see section 8 for why).")
display(Markdown("**Findings**\n\n" + "\n".join(lines)))
SUMMARY["findings"] = lines
(OUTPUT_DIR / "summary.md").write_text("# Channel/backbone abnormality-recognition comparison\n\n" + "\n".join(lines) + "\n", encoding="utf-8")
(OUTPUT_DIR / "summary.json").write_text(json.dumps(SUMMARY, indent=1, default=float))
print("outputs written to", OUTPUT_DIR)

**Findings**

- **Best overall by macro linear-probe AUROC**: **DINOv2-S / adjacent** (0.675, kNN 0.625, 18 labels evaluated).
- **Full ranking**: DINOv2-S / adjacent 0.675; DINOv2-S / windows 0.674; DINOv2-S / same 0.673; DINOv3-S / adjacent 0.665; DINOv3-S / window_edge 0.663; DALE-CT-0 (1-channel, native) 0.663; DINOv3-S / windows 0.648; DINOv3-S / same 0.647; DINOv2-S / window_edge 0.639
- **DALE-CT-0 (1-channel, native)** ranks 6 of 9 (0.663); compare this directly against the best 3-channel strategy above to judge whether the 3-channel construction helps at all over the native 1-channel method.
- 2 of 36 pairwise comparisons are significant after FDR correction (see `method_pairwise_stats.csv` for effect sizes and CIs).
- Caution: labels are CT-RATE's own (automated) annotations, not independent ground truth. This notebook measures frozen-feature probe performance only - it says nothing about compute cost or how these methods generalise to a different scanner (see `CT_Domain_Gap_Analysis.ipynb` for that).

outputs written to /content/drive/MyDrive/NHRD - Local Sample/channel_encoding_outputs
